# Brazilian tensile fracture in psammitic schist

Stress, strain and fracture analysis of a continuous-foliation schist, weak-plane spacing 2 mm, specimens
8-14 at fabric angles 0-90 degrees in 15 degree steps.

**This notebook contains no function definitions.** Every call goes into
`tools/`; the analysis sections live in `tools/analysis/`, one module per
section, and both lithology notebooks run the identical code with only the
bound lithology differing. That is what keeps the two rocks comparable.

Analysis that spans both lithologies -- the fitted strength models, mesh
convergence, the k_max sweep and the metric export -- is in
`Tensile_general_plots.ipynb`, not here.

## 1. Environment and reproducibility

In [1]:
%matplotlib inline
# Inline is the default under ipykernel, but stating it makes the stored
# outputs of a batch `nbconvert --execute` run independent of that default.

import numpy as np, pandas as pd, matplotlib

from tools import conventions, lithology, traces, failure_classification
from tools import plotting, export, strain_partitioning
from tools import analysis, output_dirs
from tools.plot_style import apply_plot_style

output_dirs.ensure(*output_dirs.ALL)   # a clean checkout carries no outputs/

TICKS = apply_plot_style()          # mandated publication style
SEED = 20260807                     # orientation-fit bootstrap

print('numpy', np.__version__, '| pandas', pd.__version__,
      '| matplotlib', matplotlib.__version__)
print('seed', SEED)

numpy 2.3.0 | pandas 2.3.0 | matplotlib 3.10.3
seed 20260807


## 2. Lithology configuration

Specimen identifiers are fixed by the experimental programme and are never
shared between lithologies. The weak-plane spacing and phase-warp amplitude
are properties of the rock, so they travel with it rather than being repeated
in each section. The assertion fails loudly if this notebook is ever pointed
at the other rock's specimens.

In [2]:
ROCK = lithology.PSAMMITIC_SCHIST
SAMPLES = ROCK.sample_ids

lithology.assert_notebook_lithology(ROCK.key, SAMPLES)

print(f'{ROCK.display_name}: samples {SAMPLES}')
print(f'fabric angles      : {ROCK.angles_deg} deg')
print(f'weak-plane spacing : {ROCK.spacing_m * 1e3:.1f} mm')
print(f'phase-warp amplitude: {ROCK.spacing_warp_amp_m * 1e3:.1f} mm')

Psammitic schist: samples (8, 9, 10, 11, 12, 13, 14)
fabric angles      : (0, 15, 30, 45, 60, 75, 90) deg
weak-plane spacing : 2.0 mm
phase-warp amplitude: 0.4 mm


## 3. Convention

Stated once, in `tools.conventions`, and used everywhere: right-handed
(x, y) in metres, disc centre at the origin, **+y the loading axis**, angles
counterclockwise from +x, orientations axial (180-degree periodic), and
tension-positive at the classifier interface.

In [3]:
for a in (0, 90):
    print(conventions.describe_end_member(a))

alpha_exp=0.0 deg -> alpha_f=0.0000 rad; trace tangent=(1.0, 0.0), normal=(6.123233995736766e-17, 1.0); trace is parallel to the horizontal diameter (perpendicular to loading).
alpha_exp=90.0 deg -> alpha_f=1.5708 rad; trace tangent=(6.123233995736766e-17, 1.0), normal=(1.0, 0.0); trace is parallel to the loading direction.


## 4. Displacement-discontinuity solve

The orthotropic moduli fields, the Hertzian contact load and the cut-cell
boundary treatment, solved by preconditioned conjugate gradient with a
BiCGSTAB fallback. Everything downstream reads the fields this produces.

In [4]:
analysis.ddm_disk.main(ROCK)


=== COMBINED PLOT (IMPROVED) SUMMARY ===
Sign convention: + tension, - compression
D=51.0 mm, t=22.0 mm, spacing=2.00 mm
Hertz contact half-width b ≈ 0.288 mm
Peak boundary pressure p_peak ≈ 350.798 MPa
Compression weight w_comp = 0.001796
Combined = σ1+ + w_comp * clamp(σyy−, -p_peak, 0)



<Figure size 1200x900 with 2 Axes>

<Figure size 1500x900 with 1 Axes>

## 5. Stress traverses across the weak planes

Radial traverses on the loading diameter, with the weak-band crossings
marked, so the stress concentration at each fabric crossing is visible
against the smooth background.

In [5]:
analysis.stress_graph.main(ROCK)

[Reference] Uniform-stress estimate for 0.63 MPa: 1286.98 N
[Calibrated] center σxx' (unit scale=1 Pa): 1.061184e-06 MPa per Pa-scale
[Calibrated] required nominal scale s0 ≈ 5.937e+05 Pa
[Result] P_failure (definition) ≈ 1267.67 N

=== VALIDATION (consistent with P_failure definition) ===
Center stress check: σxx'(center) = 0.630000 MPa (target 0.630000)
  rel err = 0.000e+00 -> PASS
Definition check: P_failure_A (unit*scale) = 1267.67 N, P_failure_B (direct) = 1267.67 N
  rel err = 0.000e+00 -> PASS
Convergence check: P_failure(2200) = 1267.67 N, P_failure(4400) = 1268.28 N
  rel diff = 4.771e-04 -> PASS
Linearity check: k=1.50, P(k*s0) = 1901.51 N, k*P = 1901.51 N
  rel err = 2.392e-16 -> PASS



<Figure size 1980x1232 with 1 Axes>

Saved: outputs/figures/psammitic_schist_stress_distribution_graph.pdf


## 6. Stress distribution over the disc

The full-disc stress field at each fabric angle.

In [6]:
analysis.stress_field.main(ROCK)


✓ Saved: outputs/figures/psammitic_schist_stress_distribution.pdf
✓ Stats: outputs/figures/combined_field_summary.csv
✓ Shared color scale: ±2.268 MPa (percentile=97.0%)


<Figure size 2205x3840 with 9 Axes>

## 7. Tensile-strain proxy

The tensile-strain proxy field and the principal direction associated with
it, smoothed nematically so the 180-degree periodicity of an orientation is
respected.

In [7]:
analysis.strain_proxy.main(ROCK)

<Figure size 2250x3840 with 9 Axes>

Saved: outputs/figures/psammitic_schist_tensile_strain_proxy.pdf
Cache dir: outputs/fields/_cache_sigma1_theta_physics_v2
Length ref (p95) = 4.685e-05
Color vmax (p99) = 1.221e-04


## 8. Principal tensile strain

Principal tensile strain panels taken from the same solved fields.

In [8]:
analysis.principal_strain.main(ROCK)

<Figure size 2205x3840 with 9 Axes>

Saved: outputs/figures/psammitic_schist_principal_tensile_strain.pdf
Cache dir: outputs/fields/_cache_solver_fields_physics_v2
Length reference magnitude (p95) = 7.928e-07


## 9. Mid-section displacement profiles

Mean horizontal displacement across the mid-section, which is the measurable
signature of the fabric opening under load.

In [9]:
analysis.displacement_profiles.main(ROCK)

Loaded: tensile_samples_data.csv
Columns: ['Rock_type', 'Angle', 'Diameter_mm', 'Thickness_mm', 'H/D_ratio', 'Area_mm^2', 'Volume_mm^3', 'Bulk_Density', 'Porosity_percent', 'Density', 'Load_(KN)', 'Tensile_strength_Mpa', 'Mass', 'Axial_strain', 'Lateral_strain', 'Modulus_of_Elasticity', 'Poisson_Ratio', 'Shear_Modulus', 'Radians', 'UCS_(Mpa)', 'Cohesion', 'Friction_Angle', 'UCS (Mpa)', 'Volume']


<Figure size 2700x1800 with 1 Axes>

Saved: outputs/figures/psammitic_schist_mid_disp_profiles.pdf
Cache dir: outputs/fields/_cache_uv_profiles_v2


## 10. Stress direction circles

Principal-direction circles around the disc boundary.

In [10]:
analysis.direction_circles.main(ROCK)

<Figure size 2205x3840 with 9 Axes>

Saved: outputs/figures/psammitic_schist_direction_circles.pdf
Color vmax (p99) = 4.461e-03 mm
Cache dir: outputs/fields/_cache_direction_circles_uv_v2


## 11. Crack growth, energy and failure fields

Energy-guided displacement-discontinuity crack growth, the four-class failure
maps, the strain-energy field and the stress-tensor glyphs.

In [11]:
analysis.crack_energy_suite.main(ROCK)

  energy colour scale (95th pct U) = 0.0364 MPa  -> outputs/tables/energy_colour_scale.csv



✓ Saved: outputs/figures/psammitic_schist_stress_tensors.pdf
✓ Saved: outputs/figures/psammitic_schist_strain_energy.pdf
✓ Stats: outputs/fields/ddm_summary_stats.csv
✓ Full fields (.npz) saved in: outputs/fields/fields_npz


[4class] mapping table: outputs/tables/classifier_mapping_psammitic_schist.csv (35 rows)
[4class] figure: outputs/figures/psammitic_schist_fourclass_map.pdf / outputs/figures/psammitic_schist_fourclass_map.png


<Figure size 2205x3840 with 8 Axes>

<Figure size 2205x3840 with 9 Axes>

## 12. Crack-path studies

Crack paths under energy and field guidance, including the four-class
failure-map guidance block.

In [12]:
analysis.crack_path_suite.main(ROCK)

[smoke_all] Running sample IDs: [8, 9, 10, 11, 12, 13, 14]

[smoke_all] === sid=8 ===
[smoke_all] sid=8 FAILED: name 'phase' is not defined

[smoke_all] === sid=9 ===
[smoke_all] sid=9 FAILED: name 'phase' is not defined

[smoke_all] === sid=10 ===
[smoke_all] sid=10 FAILED: name 'phase' is not defined

[smoke_all] === sid=11 ===
[smoke_all] sid=11 FAILED: name 'phase' is not defined

[smoke_all] === sid=12 ===


[smoke_all] sid=12 FAILED: name 'phase' is not defined

[smoke_all] === sid=13 ===
[smoke_all] sid=13 FAILED: name 'phase' is not defined

[smoke_all] === sid=14 ===
[smoke_all] sid=14 FAILED: name 'phase' is not defined
[smoke_all] No successful runs.


## 13. Fracture deviation from the foliation

How far the observed fracture departs from the fabric plane, by angle.

In [13]:
analysis.foliation_deviation.main(ROCK)

  Processing sample 8: Psammitic schist  α=0°  |  E1=33.00 GPa  E2=38.21 GPa  G12=13.42 GPa  nu12=0.23
  Processing sample 9: Psammitic schist  α=15°  |  E1=28.36 GPa  E2=32.84 GPa  G12=11.92 GPa  nu12=0.19


  Processing sample 10: Psammitic schist  α=30°  |  E1=22.24 GPa  E2=25.76 GPa  G12=9.59 GPa  nu12=0.16
  Processing sample 11: Psammitic schist  α=45°  |  E1=19.67 GPa  E2=22.78 GPa  G12=8.55 GPa  nu12=0.15


  Processing sample 12: Psammitic schist  α=60°  |  E1=20.74 GPa  E2=24.01 GPa  G12=9.02 GPa  nu12=0.15
  Processing sample 13: Psammitic schist  α=75°  |  E1=24.46 GPa  E2=28.32 GPa  G12=10.52 GPa  nu12=0.16


  Processing sample 14: Psammitic schist  α=90°  |  E1=28.50 GPa  E2=33.00 GPa  G12=11.98 GPa  nu12=0.19


Figure 1 saved → outputs/figures/schist_combined_distribution.pdf


Figure 2 saved → outputs/figures/schist_deviation_vs_angle.pdf
  deviation statistics -> outputs/tables/foliation_deviation_statistics.csv


<Figure size 750x525 with 1 Axes>

<Figure size 555x570 with 2 Axes>


Done. Both revised figures written to: outputs/figures


---

# Classification, orientation validation and energy partitioning

Diagnostics for psammitic schist, computed from the fields the sections above
exported, by the shared modules in `tools/`. Comparisons between the two rocks
live in `Tensile_general_plots.ipynb`.

This block supersedes the earlier tensile/shear/mixed classification, the
stress-profile panel built from a uniform placeholder field, and the mode counts
that were carried as literal lists. Each cell stands on its own and does not
depend on variables left behind by the sections above.

## Four-mechanism failure classification (WT / WS / MT / MS)

The utility-ratio classifier above resolves mechanism only. This section
resolves **mechanism and structural locus** together, so weak-plane opening
is no longer absorbed into a matrix category:

| | |
|---|---|
| **WT** | tensile opening along the weak plane |
| **WS** | shear sliding along the weak plane |
| **MT** | tensile cracking through the intact matrix |
| **MS** | shear cracking through the intact matrix |

Weak-plane utilities are admissible only inside the modelled activation
band, so a matrix point cannot be labelled a weak-plane failure. Mixed
mode is a secondary flag, not a fifth class.

The implementation is shared with the other lithology notebook
(`tools.failure_classification`), so the two cannot drift apart.

In [14]:
import pandas as pd, numpy as np
from tools import lithology, failure_classification, plotting, export
from tools import strain_partitioning
from tools.plot_style import apply_plot_style

TICKS = apply_plot_style()
# ROCK was bound in the configuration cell at the top of this notebook.
lithology.assert_notebook_lithology(ROCK.key, ROCK.sample_ids)

strengths = strain_partitioning.specimen_strengths()
panels = [export.classification_panel(s, strengths) for s in ROCK.sample_ids]

fourclass = pd.DataFrame([
    dict(sample=p['sample'], angle_deg=p['angle_deg'], **p['fractions'])
    for p in panels])
print(f'{ROCK.display_name}: WT/WS/MT/MS area fractions')
display(fourclass[['sample','angle_deg','WT','WS','MT','MS','none',
                   'failed_point_fraction']].round(4))

Psammitic schist: WT/WS/MT/MS area fractions


,sample,angle_deg,WT,WS,MT,MS,none,failed_point_fraction
0,8,0,0.0000,0.0000,0.0000,0.3723,0.6277,0.3723
1,9,15,0.0000,0.0059,0.0000,0.3492,0.6450,0.3550
2,10,30,0.0000,0.0795,0.0000,0.4940,0.4266,0.5734
3,11,45,0.0000,0.2078,0.0000,0.3805,0.4117,0.5883
4,12,60,0.0030,0.1936,0.0083,0.1780,0.6171,0.3829
5,13,75,0.1225,0.0000,0.0000,0.0000,0.8775,0.1225
6,14,90,0.1510,0.0000,0.0000,0.0000,0.8490,0.1510


In [15]:
paths = plotting.seven_panel_classification(
    panels, ROCK,
    f'outputs/figures/failure_mechanism_classification_{ROCK.key}_7panel')
print('written:'); [print('  ', p) for p in paths]

wt = fourclass.set_index('angle_deg')['WT']
print(f"\nWT area fraction: {wt.loc[0]:.4f} at 0 deg -> "
      f"{wt.loc[90]:.4f} at 90 deg")
print('Weak-plane opening becomes important only as the fabric rotates'
      ' into alignment with the loading diameter.')

written:
   outputs/figures/failure_mechanism_classification_psammitic_schist_7panel.pdf
   outputs/figures/failure_mechanism_classification_psammitic_schist_7panel.png

WT area fraction: 0.0000 at 0 deg -> 0.1510 at 90 deg
Weak-plane opening becomes important only as the fabric rotates into alignment with the loading diameter.


## Observed versus predicted fracture-trace orientation

Each digitized laboratory trace is paired one-to-one with its
model-derived trace for the same specimen. The primary fracture is the
connected segment of greatest arc length; orientation is fitted by total
least squares over the central half of the disc; the error is the axial
difference wrapped into [0°, 90°]. The same rule is applied to both
observed and predicted traces.

In [16]:
from tools import traces
SEED = 20260807      # orientation-fit bootstrap

rows = [traces.compare_specimen(s, seed=SEED) for s in ROCK.sample_ids]
orient = pd.DataFrame([{k: v for k, v in r.items()
                        if not k.startswith('_')} for r in rows])
display(orient[['sample','experimental_angle_deg',
                'observed_orientation_deg','observed_bootstrap_sd_deg',
                'predicted_orientation_deg',
                'abs_axial_angular_error_deg']].round(2))

,sample,experimental_angle_deg,observed_orientation_deg,observed_bootstrap_sd_deg,predicted_orientation_deg,abs_axial_angular_error_deg
0,8,0,91.74,0.82,91.32,0.43
1,9,15,85.10,0.67,85.08,0.02
2,10,30,86.01,1.98,86.80,0.80
3,11,45,92.05,0.57,90.62,1.43
4,12,60,87.11,4.78,90.95,3.83
5,13,75,96.74,0.54,89.13,7.61
6,14,90,92.62,0.67,89.88,2.74


In [17]:
agg = traces.aggregate_statistics(rows)
null = traces.null_model_statistics(rows)

print(f"n = {agg['n']}")
print(f"MAE {agg['mae_deg']:.2f} deg | RMSE {agg['rmse_deg']:.2f} | "
      f"median {agg['median_deg']:.2f} | max {agg['max_deg']:.2f}")
print(f"within 5 deg {agg['n_within_5']}/{agg['n']} | "
      f"within 10 deg {agg['n_within_10']}/{agg['n']}")
print(f"\nloading-parallel null MAE {null['mae_deg']:.2f} deg")
print('model beats the null:', null['mae_deg'] > agg['mae_deg'])
print('\nBoth observed and predicted fractures lie close to the loading'
      ' diameter, so this is a consistency check rather than a'
      ' validation of predicted orientation.')

n = 7
MAE 2.41 deg | RMSE 3.44 | median 1.43 | max 7.61
within 5 deg 6/7 | within 10 deg 7/7

loading-parallel null MAE 3.56 deg
model beats the null: True

Both observed and predicted fractures lie close to the loading diameter, so this is a consistency check rather than a validation of predicted orientation.


In [18]:
paths = plotting.seven_panel_trace_overlay(
    rows, ROCK, f'outputs/figures/fracture_trace_overlay_{ROCK.key}_7panel')
print('written:'); [print('  ', p) for p in paths]

written:
   outputs/figures/fracture_trace_overlay_psammitic_schist_7panel.pdf
   outputs/figures/fracture_trace_overlay_psammitic_schist_7panel.png


[None, None]

## Strain-energy partitioning (four-mechanism)

The energy split is computed from the same WT/WS/MT/MS classifier used
above, so the partition and the failure maps are two views of one
calculation rather than independent models.

The three regimes are uniform mean-stress offsets of +0.50, 0 and
−0.50 MPa, which is what the underlying model applies; principal
directions and deviatoric magnitudes are held fixed.

In [19]:
part = pd.DataFrame([
    strain_partitioning.partition_specimen(s, reg, strengths)
    for s in ROCK.sample_ids
    for reg in strain_partitioning.REGIME_OFFSETS_MPA])

print(f'{ROCK.display_name}: energy share by regime (% of stored energy)')
display(part.groupby('regime')[['WT_pct','WS_pct','MT_pct','MS_pct',
                                'below_threshold_pct']].mean().round(2))

output_dirs.tables()
fourclass.to_csv(f'outputs/tables/fourclass_fractions_{ROCK.key}.csv', index=False)
orient.to_csv(f'outputs/tables/orientation_validation_{ROCK.key}.csv', index=False)
part.to_csv(f'outputs/tables/strain_partition_{ROCK.key}.csv', index=False)
print('\nexported three tables to outputs/tables/')

Psammitic schist: energy share by regime (% of stored energy)


,WT_pct,WS_pct,MT_pct,MS_pct,below_threshold_pct
regime,,,,,
Extensional,7.08,9.70,1.49,47.77,33.96
Strike-Slip,5.72,9.61,0.15,47.55,36.99
Thrust,4.24,9.51,0.00,46.55,39.71



exported three tables to outputs/tables/


In [20]:
from pathlib import Path
import warnings
import numpy as np, pandas as pd

warnings.filterwarnings("ignore")
REPO = Path.cwd()          # notebooks run from the repository root, so
                           # tools/ is importable without touching sys.path

# The tables below key on the display name, so name it separately rather
# than shadowing ROCK, which is the Lithology object everything else uses.
ROCK_NAME, SLUG = ROCK.display_name, ROCK.key
SAMPLES = list(ROCK.sample_ids)

from tools import fabric_tractions as ft
print(f"{ROCK_NAME}: specimens {SAMPLES[0]}-{SAMPLES[-1]}, "
      f"{len(ft.field_files())} fields exported in total")

Psammitic schist: specimens 8-14, 14 fields exported in total


In [21]:
# Traction resolved on the foliation, and how little the principal axes move
tr  = ft.traction_table();       tr  = tr[tr.rock == ROCK_NAME].sort_values("angle_deg")
rot = ft.principal_rotation_table(); rot = rot[rot.rock == ROCK_NAME]
het = ft.heterogeneity_table();  het = het[het.rock == ROCK_NAME]

print("sigma_n on foliation (MPa)")
print("   " + "  ".join(f"{a:.0f}d={v:+6.2f}" for a, v in
                        zip(tr.angle_deg, tr.sigma_n_mean_MPa)))
print("|tau| on foliation (MPa)")
print("   " + "  ".join(f"{a:.0f}d={v:6.2f}" for a, v in
                        zip(tr.angle_deg, tr.tau_abs_mean_MPa)))
print(f"principal rotation vs 0 deg: at most {rot.rotation_median_deg.max():.2f} deg")
print(f"within-specimen orientation spread: {het.circ_sd_deg.mean():.1f} deg")

sigma_n on foliation (MPa)
   0d=-21.92  15d=-17.82  30d=-13.30  45d= -6.73  60d= -2.01  75d= +0.32  90d= +1.13
|tau| on foliation (MPa)
   0d=  3.66  15d=  5.95  30d=  8.93  45d=  8.71  60d=  6.52  75d=  2.70  90d=  2.05
principal rotation vs 0 deg: at most 2.42 deg
within-specimen orientation spread: 15.9 deg


In [22]:
# Four-mechanism classification: mechanism (tensile/shear) x locus (matrix/weak plane)
import importlib.util as ilu
spec = ilu.spec_from_file_location("mfs", REPO / "scripts/make_failure_statistics_figures.py")
mfs = ilu.module_from_spec(spec); spec.loader.exec_module(mfs)

cls = mfs.class_fractions(); cls = cls[cls.rock == ROCK_NAME].sort_values("angle_deg")
print(cls[["angle_deg", "WT", "WS", "MT", "MS", "none"]].to_string(
    index=False, float_format=lambda v: f"{v:.3f}"))
print()
print(f"matrix shear {cls.MS.iloc[0]:.2f} -> {cls.MS.iloc[-1]:.2f} with angle")
print(f"weak-plane sliding peaks at {cls.loc[cls.WS.idxmax(), 'angle_deg']:.0f} deg "
      f"({cls.WS.max():.3f})")
print(f"weak-plane opening at 90 deg: {cls.WT.iloc[-1]:.3f}")

 angle_deg    WT    WS    MT    MS  none
     0.000 0.000 0.000 0.000 0.372 0.628
    15.000 0.000 0.006 0.000 0.349 0.645
    30.000 0.000 0.079 0.000 0.494 0.427
    45.000 0.000 0.208 0.000 0.380 0.412
    60.000 0.003 0.194 0.008 0.178 0.617
    75.000 0.123 0.000 0.000 0.000 0.877
    90.000 0.151 0.000 0.000 0.000 0.849

matrix shear 0.37 -> 0.00 with angle
weak-plane sliding peaks at 45 deg (0.208)
weak-plane opening at 90 deg: 0.151


In [23]:
# Connectivity of the elevated strain-energy region: corridor or separate lobes
from tools import energy_localization as el
t = el.localization_table(); t = t[t.rock == ROCK_NAME].sort_values("angle_deg")
print(t[["angle_deg", "n_components", "elongation", "corridor"]].to_string(
    index=False, float_format=lambda v: f"{v:.2f}"))
print()
print(f"corridor forms at {int(t.corridor.sum())} of {len(t)} orientations")

 angle_deg  n_components  elongation  corridor
      0.00             1        4.79      True
     15.00             1        4.63      True
     30.00             2        1.30     False
     45.00             2        1.17     False
     60.00             2        1.25     False
     75.00             2        1.46     False
     90.00             1        4.62      True

corridor forms at 3 of 7 orientations


In [24]:
# Figures for this lithology are written by the shared scripts, which cover
# both rocks in one pass and therefore run once, in the general notebook.
# Here we only confirm that this lithology's panels are present.
from tools import figure_scripts

figure_scripts.report_expected((
    f"{SLUG}_stress_tensors.pdf",
    f"{SLUG}_stress_distribution_graph.pdf",
    f"fig_S_energy_localization_{SLUG}.pdf",
))

  ok      psammitic_schist_stress_tensors.pdf
  ok      psammitic_schist_stress_distribution_graph.pdf
  ok      fig_S_energy_localization_psammitic_schist.pdf


[]